# Data Preparation & Noise Injection
Load ImageNet-100, inspect samples, apply corruption strategies, and build train/val/test dataloaders.

In [ ]:
from pathlib import Path
import yaml
import torch
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

from src.dataset import ImageNet100Dataset, NoiseInjector, get_dataloaders

In [ ]:
config_path = Path('../configs/config.yaml')
with config_path.open('r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config

In [ ]:
root = Path(config['dataset']['root'])
if not root.exists():
    print(f'ImageNet-100 root not found at: {root.resolve()}')
    print('Create class folders under data/raw before training.')
else:
    class_dirs = sorted([p.name for p in root.iterdir() if p.is_dir()])
    print(f'Found {len(class_dirs)} class folders.')
    print('Sample classes:', class_dirs[:10])

In [ ]:
injector = NoiseInjector()
injector

In [ ]:
dataset = ImageNet100Dataset(root=config['dataset']['root'], image_size=config['dataset']['image_size'])
idx = torch.randperm(len(dataset))[:8]
clean_images = torch.stack([dataset[int(i)][0] for i in idx])

grid = make_grid(clean_images, nrow=4, normalize=True)
plt.figure(figsize=(10, 6))
plt.imshow(grid.permute(1, 2, 0))
plt.title('8 Clean Samples')
plt.axis('off')
plt.show()

In [ ]:
sample = clean_images[0]
gaussian = injector.add_gaussian_noise(sample, std=config['noise']['gaussian_std'])
salt_pepper = injector.add_salt_pepper(sample, prob=config['noise']['salt_pepper_prob'])
occluded = injector.add_occlusion(sample, patch_size=config['noise']['occlusion_size'])

compare = torch.stack([sample, gaussian, salt_pepper, occluded])
titles = ['Clean', 'Gaussian', 'Salt & Pepper', 'Occlusion']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    ax.imshow(compare[i].permute(1, 2, 0).detach().cpu().numpy(), vmin=-2, vmax=2)
    ax.set_title(titles[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
train_loader, val_loader, test_loader = get_dataloaders(
    root=config['dataset']['root'],
    image_size=config['dataset']['image_size'],
    batch_size=config['vae']['batch_size'],
    train_split=config['dataset']['train_split'],
    num_workers=config['training']['num_workers'],
    noisy=True,
    noise_type='gaussian',
    noise_params={'std': config['noise']['gaussian_std']}
)

noisy_batch, clean_batch, labels = next(iter(train_loader))
print('Noisy batch shape:', noisy_batch.shape)
print('Clean batch shape:', clean_batch.shape)
print('Labels shape:', labels.shape)

## Summary
ImageNet-100 loading, corruption injection, visualization, and dataloader preparation are now ready for model training stages.